### LightGCN ID Embedding — 图卷积协同嵌入
 方法概述：LightGCN + BPR 协同 ID Embedding

使用 **LightGCN（轻量图卷积网络）** 从用户-物品交互图中学习结构感知的协同嵌入。


### 配置
| 参数 | 值 | 说明 |
|------|-----|------|
| EMBED_DIM | 512 | 嵌入维度 |
| N_LAYERS | 3 | 图卷积层数（感受野 = 3-hop 邻居） |
| EPOCHS | 30 | 训练轮数（wd=0 后收敛快） |
| BATCH_SIZE | 2048 | 每批 2048 个 (u,i) 对 |
| LR | 1e-3 | Adam 学习率 |


In [6]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, json, os
from scipy.sparse import csr_matrix, diags
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

TRAIN_CSV    = "final/train.csv"
ITEM2ID_PATH = "final/item2id.json"
OUTPUT_DIR   = "final/features"
EMBED_DIM    = 512
N_LAYERS     = 3
LR           = 1e-3
EPOCHS       = 30
BATCH_SIZE   = 2048
L2_REG       = 0     # 必须为 0！Xavier init 权重极小，weight_decay 会抵消所有梯度
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}  Embed: {EMBED_DIM}d  Layers: {N_LAYERS}")

Device: cuda  Embed: 512d  Layers: 3


###  用户-物品交互建模为二部图，通过多层图卷积传播协同信号：
- 节点：20,030 用户 + 43,528 物品 = 63,558 个节点
- 边：647,950 条交互（用户↔物品，边权=1）
- 学习每个节点（用户/物品）的 512 维稠密向量，使得交互过的用户-物品对在嵌入空间中距离更近


In [ ]:
train = pd.read_csv(TRAIN_CSV)
train["user_id"] = train["user_id"].astype(int)
train["item_id"] = train["item_id"].astype(int)

with open(ITEM2ID_PATH) as f:
    item2id = json.load(f)
n_items = len(item2id)
n_users = train["user_id"].max() + 1
print(f"Users: {n_users}  Items: {n_items}  Interactions: {len(train)}")
print(f"Sparsity: {len(train)/(n_users*n_items)*100:.3f}%") # 稀疏图

Users: 20030  Items: 43528  Interactions: 647950
Sparsity: 0.074%


In [ ]:
user_ids = train["user_id"].values
item_ids = train["item_id"].values

N = n_users + n_items
row = np.concatenate([user_ids, item_ids + n_users])
col = np.concatenate([item_ids + n_users, user_ids])
data = np.ones(2 * len(train), dtype=np.float32)
A = csr_matrix((data, (row, col)), shape=(N, N))

# 邻接矩阵归一化
degrees = np.array(A.sum(axis=1)).flatten()
deg_inv_sqrt = np.power(degrees, -0.5)
deg_inv_sqrt[np.isinf(deg_inv_sqrt)] = 0.0
D_inv_sqrt = diags(deg_inv_sqrt)
A_norm = D_inv_sqrt @ A @ D_inv_sqrt
print(f"Adjacency: {A_norm.shape}  nnz={A_norm.nnz}")

Building normalized adjacency matrix...
Adjacency: (63558, 63558)  nnz=1275240


### 图卷积传播（LightGCN）
LightGCN 图协同过滤模型的初始化 + 训练前梯度自检模块，核心作用是在用户 - 物品交互图上预训练高质量的 ID 嵌入。

仅保留**线性邻域聚合**：

$$\mathbf{E}^{(k+1)} = \tilde{A} \cdot \mathbf{E}^{(k)}$$

其中 $\tilde{A} = D^{-1/2} A D^{-1/2}$ 是对称归一化的邻接矩阵：
- $A$：用户-物品二部图的邻接矩阵（无向边，对称）
- $D^{-1/2}$：度的 -0.5 次方对角矩阵，用于归一化，防止高度节点主导传播

每一层传播等价于：节点的嵌入 = 其邻居节点嵌入的（归一化）均值。

最终嵌入取 **所有层的均值池化**（mean-pooling）：

$$\mathbf{E}_{\text{final}} = \frac{1}{K+1} \sum_{k=0}^{K} \mathbf{E}^{(k)}$$


In [ ]:
# Embedding 初始化
E0_user = nn.Embedding(n_users, EMBED_DIM)
E0_item = nn.Embedding(n_items, EMBED_DIM)
nn.init.xavier_uniform_(E0_user.weight)
nn.init.xavier_uniform_(E0_item.weight)

# 稀疏邻接矩阵转换为 PyTorch 稀疏张量
def sparse_to_torch(sp_mat):
    coo = sp_mat.tocoo()
    indices = torch.LongTensor(np.vstack([coo.row, coo.col]))
    values = torch.FloatTensor(coo.data)
    return torch.sparse_coo_tensor(indices, values, sp_mat.shape)

# LightGCN 多层传播函数
def lightgcn_propagate(e0_user, e0_item, A_sp):
    e0 = torch.cat([e0_user.weight, e0_item.weight], dim=0) # 拼接
    emb_list = [e0]
    cur = e0
    # 领域加权聚合 
    for _ in range(N_LAYERS):
        cur = torch.sparse.mm(A_sp, cur)
        emb_list.append(cur)
    # 多层均值池化
    final = torch.stack(emb_list, dim=0).mean(dim=0)
    return final[:n_users], final[n_users:] # 用户嵌入 与 物品嵌入

# 移至 GPU + 创建 Optimizer
E0_user = E0_user.to(DEVICE)
E0_item = E0_item.to(DEVICE)
A_sparse = sparse_to_torch(A_norm).to(DEVICE)

optimizer = torch.optim.Adam(
    list(E0_user.parameters()) + list(E0_item.parameters()),
    lr=LR, weight_decay=L2_REG
)

pos_pairs = list(zip(user_ids, item_ids))
item_set_per_user = {}
for u, i in zip(user_ids, item_ids):
    item_set_per_user.setdefault(u, set()).add(i)

print(f"Graph nodes: {N:,}  Init done.")
print(f"Propagation: {N_LAYERS}-layer mean-pooling")
print(f"Training {EPOCHS} epochs, {len(pos_pairs):,} pairs, batch={BATCH_SIZE}")
print(f"E0_user device: {E0_user.weight.device}")
print(f"A_sparse device: {A_sparse.device}")

# 梯度自检：验证 optimizer 参数与 forward 参数一致
print("\n[Gradient check]")
_check_u = torch.LongTensor([pos_pairs[0][0]]).to(DEVICE)
_check_i = torch.LongTensor([pos_pairs[0][1]]).to(DEVICE)
_check_j = torch.randint(0, n_items, (1,)).to(DEVICE)
_ug = E0_user(_check_u)
_pg = E0_item(_check_i)
_ng = E0_item(_check_j)
_ps = (_ug * _pg).sum(dim=1)
_ns = (_ug * _ng).sum(dim=1)
_cl = -F.logsigmoid(_ps - _ns).mean()
_cl.backward()
_gn_u = E0_user.weight.grad.norm().item()
_gn_i = E0_item.weight.grad.norm().item()
optimizer.zero_grad()
print(f"  pos={_ps.item():.4f}  neg={_ns.item():.4f}  loss={_cl.item():.4f}")
print(f"  grad norm: user={_gn_u:.4f}  item={_gn_i:.4f}")
if _gn_u > 0 and _gn_i > 0:
    print("  ✓ Gradients flowing — training will work")
else:
    print("  ✗ GRADIENT IS ZERO — STOP and re-run from Cell 1!")
print()

Graph nodes: 63,558  Init done.
Propagation: 3-layer mean-pooling
Training 30 epochs, 647,950 pairs, batch=2048
E0_user device: cuda:0
A_sparse device: cuda:0

[Gradient check]
  pos=-0.0013  neg=0.0012  loss=0.6944
  grad norm: user=0.1054  item=0.1570
  ✓ Gradients flowing — training will work



### 3. BPR（贝叶斯个性化排序）排序训练循环

基本假设：用户对交互过的物品的偏好应高于未交互的物品。

$$\mathcal{L}_{\text{BPR}} = -\frac{1}{|\mathcal{B}|} \sum_{(u,i,j) \in \mathcal{B}} \ln \sigma(\hat{y}_{ui} - \hat{y}_{uj})$$

In [ ]:
# BPR 训练（仅在原始嵌入上，不做图传播）
# 图传播在训练完成后一次性执行
for epoch in range(1, EPOCHS + 1):
    np.random.shuffle(pos_pairs)
    total_loss, n_batch = 0, 0
    pbar = tqdm(range(0, len(pos_pairs), BATCH_SIZE),
                desc=f"Epoch {epoch:2d}/{EPOCHS}", leave=True)
    for start in pbar:
        # 采样一个批次的正样本
        batch = pos_pairs[start:start + BATCH_SIZE]
        u_ids = torch.LongTensor([p[0] for p in batch]).to(DEVICE)
        i_ids = torch.LongTensor([p[1] for p in batch]).to(DEVICE)

        # 负采样
        j_ids = []
        for u in u_ids.cpu().numpy():
            neg = np.random.randint(0, n_items)
            while neg in item_set_per_user.get(int(u), set()):
                neg = np.random.randint(0, n_items)
            j_ids.append(neg)
        j_ids = torch.LongTensor(j_ids).to(DEVICE)

        # 直接从原始 Embedding 取（训练时不传播，传播放在训练后）
        # 前向计算
        u_emb = E0_user(u_ids)
        pos_emb = E0_item(i_ids)
        neg_emb = E0_item(j_ids)

        pos_score = (u_emb * pos_emb).sum(dim=1)
        neg_score = (u_emb * neg_emb).sum(dim=1)
        loss = -F.logsigmoid(pos_score - neg_score).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batch += 1
        pbar.set_postfix({"loss": f"{total_loss/n_batch:.4f}"})

    print(f"  Epoch {epoch:3d} avg_loss={total_loss/n_batch:.4f}")

Epoch  1/30: 100%|██████████| 317/317 [00:08<00:00, 35.51it/s, loss=0.6884]


  Epoch   1 avg_loss=0.6884


Epoch  2/30: 100%|██████████| 317/317 [00:08<00:00, 38.38it/s, loss=0.5535]


  Epoch   2 avg_loss=0.5535


Epoch  3/30: 100%|██████████| 317/317 [00:08<00:00, 38.54it/s, loss=0.3495]


...

  Epoch  28 avg_loss=0.0030


Epoch 29/30: 100%|██████████| 317/317 [00:09<00:00, 32.20it/s, loss=0.0027]


  Epoch  29 avg_loss=0.0027


Epoch 30/30: 100%|██████████| 317/317 [00:09<00:00, 31.83it/s, loss=0.0025]

  Epoch  30 avg_loss=0.0025


In [ ]:
# 执行图传播，提取最终物品嵌入
print("Extracting final item embeddings...")
with torch.no_grad():
    _, final_items = lightgcn_propagate(E0_user, E0_item, A_sparse)
    item_emb_np = final_items.cpu().numpy()

out_path = f"{OUTPUT_DIR}/item_id_lightgcn_512.npy"
np.save(out_path, item_emb_np)
print(f"Saved: {out_path}")
print(f"Shape: {item_emb_np.shape}  Min: {item_emb_np.min():.4f}  Max: {item_emb_np.max():.4f}")
print(f"Zero rows: {(item_emb_np.sum(1)==0).sum()} (expect 0)")

Extracting final item embeddings...
Saved: final/features/item_id_lightgcn_512.npy
Shape: (43528, 512)  Min: -0.5095  Max: 0.4557
Zero rows: 0 (expect 0)
